In [ ]:
#@title Connect to the release { display-mode: "form" }
import os
import sys

try:
    from google.colab import drive as google_drive
except ImportError:
    pass
else:
    google_drive.mount("/content/drive", force_remount=False)
CODE = os.environ.get(
    "ZHONGDB_CODE",
    "/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace/code",
)
if CODE not in sys.path:
    sys.path.insert(0, CODE)

for name in ("database", "dprime", "drive", "graph", "position", "sql"):
    sys.modules.pop(name, None)
import drive
import graph

db = drive.setup()
db.schema()

In [ ]:
info = db.load_file("Imaging_Exp_info.npy")
info.keys()

In [ ]:
info['naive_test1']

In [ ]:
# we have neural data for these experiments
experiments = {
    "supervised": (
        "sup_train1_before_learning",
        "sup_train1_after_learning",
        "sup_test1",
        "sup_train2_before_learning",
        "sup_train2_after_learning",
        "sup_test2",
        "sup_test3",
    ),
    "unsupervised": (
        "unsup_train1_before_learning",
        "unsup_train1_after_learning",
        "unsup_test1",
        "unsup_train2_before_learning",
        "unsup_train2_after_learning",
        "unsup_test2",
        "unsup_test3",
    ),
    "grating": (
        "train1_before_grating",
        "train1_after_grating",
        "test1_before_grating",
        "test1_after_grating",
        "test2_before_grating",
        "test2_after_grating",
    ),
    "naive": (
        "naive_test1",
        "naive_test2",
        "naive_test3",
    ),
}

experiments

In [ ]:
# we have neural data for these mice
mice = {
    "supervised": ['TX108', 'TX109', 'TX60', 'TX61', 'VR2'],
    "unsupervised": ['DR10', 'DR15', 'TX104', 'TX105', 'TX119', 'TX123', 'TX83', 'TX85', 'TX88'],
    "grating": ['LZ13', 'LZ16', 'TX139'],
    "naive": ['TX124', 'TX140'],
}

In [ ]:
db.query("""
    SELECT *
    FROM recordings
    WHERE mouse = 'TX108'
    ORDER BY date, block
""")

In [ ]:
mice_recordings = {
    category: {
        mouse: db.query("""
            SELECT * FROM recordings
            WHERE mouse = ?
            ORDER BY date, block
        """, [mouse]).to_dict("records")
        for mouse in elements
    }
    for category, elements in mice.items()
}
{category: len(recordings) for category, recordings in mice_recordings.items()}

In [ ]:
mice_recordings['supervised']['TX108']

In [ ]:
def files(recordings):
    return [
        db.query("""
            SELECT filename
            FROM recording_files
            WHERE recording_id = ?
            ORDER BY layer, experiment
        """, [recording["recording_id"]])["filename"].tolist()
        for recording in recordings
    ]

In [ ]:
files(mice_recordings['supervised']['TX108'])

In [ ]:
files(mice_recordings['naive']['TX124'])

In [ ]:
files(mice_recordings['naive']['TX140'])

In [ ]:
train1_labels = {
    "supervised": (
        "sup_train1_before_learning",
        "sup_train1_after_learning",
    ),
    "unsupervised": (
        "unsup_train1_before_learning",
        "unsup_train1_after_learning",
    ),
    "grating": (
        "train1_before_grating",
        "train1_after_grating",
    ),
    "naive": (
        "naive_test1",
    ),
}

In [ ]:
mice_recordings

In [ ]:
def select_recordings(category, label):
    return db.query("""
        SELECT DISTINCT r.*
        FROM memberships AS membership
        JOIN recordings AS r USING (recording_id)
        JOIN mice AS mouse_info ON mouse_info.mouse = r.mouse
        WHERE membership.experiment = ?
          AND mouse_info.primary_cohort = ?
          AND r.has_full_neural
        ORDER BY r.mouse, r.date, r.block
    """, [label, category]).to_dict("records")


train1_recordings = {
    category: {
        label: select_recordings(category, label)
        for label in labels
    }
    for category, labels in train1_labels.items()
}
train1_recordings

In [ ]:
{ category: recording.keys() for (category, recording) in train1_recordings.items() }

In [ ]:
files(train1_recordings['naive']['naive_test1'])

In [ ]:
file_map = {
    category: {
        label: files(recordings)
        for (label, recordings) in recording_map.items()
    }
    for (category, recording_map) in train1_recordings.items()
}
file_map

In [ ]:
file_map['supervised']

In [ ]:
file_map['unsupervised']

In [ ]:
file_map['grating']

In [ ]:
file_map['naive']

In [ ]:
train1_labels

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def dropdown(options, value, description, onchange = None):
    widget = widgets.Dropdown(
        options=options,
        value=value,
        description=description,
    )
    if onchange is not None:
      widget.observe(onchange, names="value")
    return widget

def dict_array_selector(dict_array):
  keys = list(dict_array.keys())

  def on_key_change(change):
      new_key = change["new"]
      value_dropdown.options = dict_array[new_key]
      if dict_array[new_key]:
          value_dropdown.value = dict_array[new_key][0]

  key_dropdown = dropdown(
      options=keys,
      value=keys[0],
      description="Key:",
      onchange=on_key_change
  )

  value_dropdown = dropdown(
      options=dict_array[keys[0]],
      value=dict_array[keys[0]][0],
      description="Value:"
  )

  return widgets.VBox([key_dropdown, value_dropdown])

def dict_array_selector_values(dict_array_selector):
  return {
      dict_array_selector.children[0].value: dict_array_selector.children[1].value
  }

In [ ]:
sup_train1 = dict_array_selector(file_map['supervised'])

In [ ]:
#@title Join neural and behavior by frame (DuckDB SQL)
import duckdb
import numpy as np
import pandas as pd
from position import align_trailing_behavior_frames

# Load one recording.
choice = dict_array_selector_values(sup_train1)
experiment, selected_files = next(iter(choice.items()))
neural_filename = next(
    name for name in selected_files if name.endswith("_neural_data.npy")
)
recording_id = neural_filename.removesuffix("_neural_data.npy")
recording_row = db.query(
    "SELECT * FROM recordings WHERE recording_id = ?",
    [recording_id],
).iloc[0]
behavior = db.load(recording_id, "behavior", experiment=experiment)
svd = db.load(recording_id, "reduced_neural")

U = np.asarray(svd["U"])
V = np.asarray(svd["V"])
moving = (
    np.asarray(behavior["ft_isMoving"])
    if "ft_isMoving" in behavior
    else np.asarray(behavior["ft_move"]) > 0
)

# Synchronize the two time series before doing relational joins.
V_by_frame, aligned, report = align_trailing_behavior_frames(
    V.T,
    {
        "trial_id": behavior["ft_trInd"],
        "position_dm": behavior["ft_Pos"],
        "is_moving": moving,
        "run_speed": behavior["ft_RunSpeed"],
    },
    max_trailing_behavior_frames=3,
)

# Three small tables, joined through explicit keys:
# frame -> trial_id -> wall_name -> stim_id
trial_raw = np.asarray(aligned["trial_id"], dtype=float)
integer_trial = np.isfinite(trial_raw) & np.isclose(trial_raw, np.rint(trial_raw))
trial_id = pd.Series(trial_raw).where(integer_trial).round().astype("Int64")

neural_frames = pd.DataFrame(
    V_by_frame[:, :8],
    columns=[f"PC{i + 1}" for i in range(8)],
)
neural_frames.insert(0, "frame", np.arange(len(V_by_frame)))

behavior_frames = pd.DataFrame({
    "frame": np.arange(len(V_by_frame)),
    "trial_id": trial_id,
    "position_dm": np.asarray(aligned["position_dm"], dtype=float),
    "is_moving": np.asarray(aligned["is_moving"], dtype=bool),
    "run_speed": np.asarray(aligned["run_speed"], dtype=float),
})

trial_lookup = pd.DataFrame({
    "trial_id": np.arange(len(behavior["WallName"])),
    "wall_name": np.asarray(behavior["WallName"]),
})

stimulus_lookup = pd.DataFrame({
    "wall_name": np.asarray(behavior["UniqWalls"]),
    "stim_id": pd.to_numeric(
        pd.Series(np.asarray(behavior["stim_id"])),
        errors="coerce",
    ).astype("Int64"),
})

joined = duckdb.sql("""
    SELECT
        n.frame,
        t.trial_id IS NOT NULL AS valid_trial,
        COALESCE(t.trial_id, -1)::BIGINT AS trial_id,
        b.position_dm,
        b.position_dm / 10.0 AS position_m,
        b.is_moving,
        b.run_speed,
        t.wall_name,
        COALESCE(s.stim_id, -1)::INTEGER AS stim_id,
        n.* EXCLUDE (frame)
    FROM neural_frames AS n
    JOIN behavior_frames AS b USING (frame)
    LEFT JOIN trial_lookup AS t ON b.trial_id = t.trial_id
    LEFT JOIN stimulus_lookup AS s ON t.wall_name = s.wall_name
    ORDER BY n.frame
""").df()

assert len(joined) == report.neural_frames
assert joined["frame"].is_unique

print("Experiment:", experiment)
print("Recording:", recording_id)
print(report)
print("Joined:", joined.shape)
print("Frames without a valid trial:", int((~joined["valid_trial"]).sum()))
joined.head()


In [ ]:
#@title Count moving texture frames with SQL
stimulus_counts = duckdb.sql("""
    SELECT
        COUNT(*) FILTER (WHERE stim_id = 2) AS leaf_frames,
        COUNT(*) FILTER (WHERE stim_id = 0) AS circle_frames
    FROM joined
    WHERE valid_trial
      AND is_moving
      AND position_m >= 0
      AND position_m < 4
""").df().iloc[0]

leaf_frame_count = int(stimulus_counts["leaf_frames"])
circle_frame_count = int(stimulus_counts["circle_frames"])

print("Leaf frames:", leaf_frame_count)
print("Circle frames:", circle_frame_count)
assert leaf_frame_count == 3152
assert circle_frame_count == 2712


In [ ]:
from dprime import prepare_session_trials

retinotopy = db.load(recording_id, "retinotopy")

trial_data = prepare_session_trials(
    behavior,
    svd,
    retinotopy,
    area="V1",
    n_features=12,
    n_position_bins=18,
    movement_rule="moving_only",
    mouse_id=recording_row["mouse"],
    recording_id=recording_id,
)

print("trial_features:", trial_data["trial_features"].shape)
print("run_speed:", trial_data["run_speed"].shape)
print("labels:", trial_data["labels"].shape)
print("alignment:", trial_data["alignment"])

In [ ]:
#@title Verify the neural-behavior join end to end
import numpy as np
from pprint import pprint

from position import (
    align_trailing_behavior_frames,
    bin_trial_features,
    decimeters_to_meters,
)
from dprime import area_transform

U_check = np.asarray(svd["U"])
V_check = np.asarray(svd["V"])
moving_check = (
    np.asarray(behavior["ft_isMoving"])
    if "ft_isMoving" in behavior
    else np.asarray(behavior["ft_move"]) > 0
)

V_frames_check, behavior_check, alignment_check = align_trailing_behavior_frames(
    V_check.T,
    {
        "position_dm": behavior["ft_Pos"],
        "trial_id": behavior["ft_trInd"],
        "is_moving": moving_check,
        "run_speed": behavior["ft_RunSpeed"],
    },
    max_trailing_behavior_frames=3,
)

position_m_check = decimeters_to_meters(behavior_check["position_dm"])
trial_raw_check = np.asarray(behavior_check["trial_id"], dtype=float)
valid_join_frames = (
    np.asarray(behavior_check["is_moving"], dtype=bool)
    & np.isfinite(position_m_check)
    & np.isfinite(trial_raw_check)
    & (position_m_check >= 0.0)
    & (position_m_check <= 6.0)
)

# Independently reconstruct the trial x position binning.
edges_check = np.linspace(0.0, 6.0, 19)
trial_ids_check, binned_v_check, counts_check = bin_trial_features(
    V_frames_check,
    position_m_check,
    trial_raw_check,
    edges_check,
    valid_mask=valid_join_frames,
)
speed_ids_check, speed_check, speed_counts_check = bin_trial_features(
    behavior_check["run_speed"],
    position_m_check,
    trial_raw_check,
    edges_check,
    valid_mask=valid_join_frames,
)

# Independently reconstruct the V1 feature projection.
transform_check = area_transform(
    U_check,
    retinotopy["iarea"],
    "V1",
    n_features=12,
)
features_check = np.einsum(
    "tbp,pk->tbk",
    binned_v_check,
    transform_check,
    optimize=True,
).astype(np.float32)

# Independently reconstruct trial labels.
wall_names_check = np.asarray(behavior["WallName"])
wall_by_trial_check = wall_names_check[trial_ids_check]
wall_to_role_check = {}
for wall, role in zip(
    np.asarray(behavior["UniqWalls"]),
    np.asarray(behavior["stim_id"]),
):
    try:
        if np.isfinite(role):
            wall_to_role_check[str(wall)] = int(role)
    except TypeError:
        pass

labels_check = np.asarray(
    [wall_to_role_check.get(str(wall), -1) for wall in wall_by_trial_check],
    dtype=np.int16,
)

# Frame-level label support, safely excluding NaN/out-of-range ft_trInd.
integer_trial_frames = np.isfinite(trial_raw_check) & np.isclose(
    trial_raw_check,
    np.rint(trial_raw_check),
)
frame_trial_ids_check = np.full(len(trial_raw_check), -1, dtype=np.int64)
frame_trial_ids_check[integer_trial_frames] = np.rint(
    trial_raw_check[integer_trial_frames]
).astype(np.int64)
in_range_trial_frames = (
    integer_trial_frames
    & (frame_trial_ids_check >= 0)
    & (frame_trial_ids_check < len(wall_names_check))
)
frame_roles_check = np.full(len(trial_raw_check), -1, dtype=np.int16)
frame_walls_check = np.full(len(trial_raw_check), None, dtype=object)
frame_walls_check[in_range_trial_frames] = wall_names_check[
    frame_trial_ids_check[in_range_trial_frames]
]
frame_roles_check[in_range_trial_frames] = [
    wall_to_role_check.get(str(wall), -1)
    for wall in frame_walls_check[in_range_trial_frames]
]
texture_moving_check = (
    valid_join_frames
    & (position_m_check < 4.0)
    & in_range_trial_frames
)
leaf_frame_count = int(np.count_nonzero(
    texture_moving_check & (frame_roles_check == 2)
))
circle_frame_count = int(np.count_nonzero(
    texture_moving_check & (frame_roles_check == 0)
))

# Hard assertions: any failure stops here with a precise traceback.
assert recording_id == "TX108_2023_03_13_1"
assert experiment == "sup_train1_before_learning"
assert U_check.ndim == 2 and V_check.ndim == 2
assert U_check.shape[0] == V_check.shape[0]
assert U_check.shape[1] == len(np.asarray(retinotopy["iarea"]))
assert V_check.shape[1] == alignment_check.neural_frames
assert alignment_check.behavior_frames - alignment_check.neural_frames == alignment_check.dropped_trailing_behavior_frames
assert alignment_check.dropped_trailing_behavior_frames <= 3
assert all(len(values) == alignment_check.neural_frames for values in behavior_check.values())
assert np.array_equal(trial_ids_check, speed_ids_check)
assert np.array_equal(counts_check, speed_counts_check)
assert np.array_equal(trial_ids_check, trial_data["trial_id"])
assert np.array_equal(counts_check, trial_data["frame_counts"])
assert np.allclose(speed_check[:, :, 0], trial_data["run_speed"], equal_nan=True)
assert np.allclose(features_check, trial_data["trial_features"], equal_nan=True, rtol=1e-5, atol=1e-5)
assert np.array_equal(wall_by_trial_check, trial_data["wall_name"])
assert np.array_equal(labels_check, trial_data["labels"])
assert trial_data["trial_features"].shape == (210, 18, 12)
assert trial_data["run_speed"].shape == (210, 18)
assert trial_data["labels"].shape == (210,)
assert trial_data["frame_counts"].shape == (210, 18)
assert len(trial_data["position_centers_m"]) == 18
assert np.all(trial_data["frame_counts"] >= 0)
assert leaf_frame_count > 0 and circle_frame_count > 0
assert 0 in set(trial_data["labels"]) and 2 in set(trial_data["labels"])

label_values, label_counts = np.unique(
    trial_data["labels"],
    return_counts=True,
)
verification_summary = {
    "status": "ALL CHECKS PASSED",
    "recording_id": recording_id,
    "experiment": experiment,
    "U_components_x_neurons": U_check.shape,
    "V_components_x_frames": V_check.shape,
    "alignment": alignment_check.to_dict(),
    "trial_features": trial_data["trial_features"].shape,
    "run_speed": trial_data["run_speed"].shape,
    "frame_counts": trial_data["frame_counts"].shape,
    "label_counts": dict(zip(label_values.tolist(), label_counts.tolist())),
    "moving_texture_leaf_frames": leaf_frame_count,
    "moving_texture_circle_frames": circle_frame_count,
    "unassigned_frame_count": int(np.count_nonzero(~in_range_trial_frames)),
    "empty_trial_position_bins": int(np.count_nonzero(trial_data["frame_counts"] == 0)),
    "finite_neural_fraction": float(np.mean(np.isfinite(trial_data["trial_features"]))),
    "finite_speed_fraction": float(np.mean(np.isfinite(trial_data["run_speed"]))),
}
pprint(verification_summary)
